Field() - lets you give additional information about a field.

In [ ]:
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str = Field(description="Name of the product")
    price: float = Field(description="Price of the product")
    category: str = Field(description="Product category")

### Validation

This means:
- rating >= 1
- rating <= 5

In [ ]:
class ProductRating(BaseModel):
    rating: int = Field(ge=1, le=5)

### Literal
When you want only specific choices:
Allowed:
- positive ✅
- negative ✅
- neutral ❌
- random ❌
This is extremely useful for classification.

In [ ]:
from typing import Literal

class Review(BaseModel):
    sentiment: Literal["positive", "negative"]

### Lists
- You can also request arrays/lists.

In [ ]:
class Review(BaseModel):
    sentiment: Literal["positive", "negative"]
    key_points: list[str]

class Review(BaseModel):
    sentiment: Literal["positive", "negative"]
    key_points: list[str]

### Structured Output Directly from the Model

In [ ]:
# Now an important problem.

# Gemini returns:

# "Python is a programming language..."

# But your application might need:

# {
#     "answer": "...",
#     "difficulty": "easy"
# }

# That's where output parsing / structured output comes in.

In [2]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

class TopicInfo(BaseModel):
    summary: str = Field(description="Short explanation")
    difficulty: str = Field(description="easy, medium, or hard")
    key_points: list[str]

structured_llm = llm.with_structured_output(TopicInfo)

response = structured_llm.invoke(
    "Explain RAG"
)

print(response)

summary='Retrieval-Augmented Generation (RAG) is a technique that improves the accuracy and reliability of generative AI models by fetching facts from an external knowledge base before generating a response.' difficulty='medium' key_points=['Combines information retrieval with text generation', 'Reduces hallucinations in Large Language Models', 'Allows models to access up-to-date or private data without retraining']


In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

class Product(BaseModel):
    name: str = Field(description="Product name")
    price: float = Field(description="Product price")
    category: str = Field(description="Product category")

structured_llm = llm.with_structured_output(Product)

result = structured_llm.invoke(
    "The Yamaha F280 costs around 9000 rupees and is an acoustic guitar."
)

print(result)

name='Yamaha F280' price=9000.0 category='acoustic guitar'


### Prompt + structured output

In [4]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an AI tutor."
    ),
    (
        "human",
        "Explain {topic}."
    )
])

class TopicInfo(BaseModel):
    summary: str = Field(description="One line short explanation")
    difficulty: str = Field(description="easy, medium, or hard")
    key_points: list[str]

structured_llm = llm.with_structured_output(
    TopicInfo
)

chain = prompt | structured_llm

response = chain.invoke({
    "topic": "FIFO"
})

print(response)

summary='FIFO stands for First-In, First-Out, a method used for data processing and inventory management where the first element is processed first.' difficulty='easy' key_points=['Operates on the principle that the oldest entry is processed first.', 'Commonly implemented using a queue data structure.', 'Used in inventory valuation to sell oldest stock first.']
